# Classification task with CWT and CNN model

This model is based on ["Real-Time Stress Detection via Photoplethysmogram Signals: Implementation of a Combined Continuous Wavelet Transform and Convolutional Neural Network on Resource-Constrained Microcontrollers"](https://ieeexplore.ieee.org/document/10668302) 

In [ ]:
%matplotlib widget
%load_ext tensorboard
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias

import tensorflow as tf
import datetime

from tensorflow import keras
from keras import layers, models, Input
from keras.src.legacy.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.optimizers import Adam

from scipy.signal import resample, medfilt
from skimage.transform import resize
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, LeaveOneOut, GroupKFold, StratifiedKFold, KFold

DATASET = "../../stressid-dataset"
LABELS_SEPARATOR = ","
LABELS = "../labels.csv"
RESULTS_DIR = "./Results"
LOGS_DIR = f"{RESULTS_DIR}/logs"
DATA_SEPARATOR = ","
DATA_ECG = f"{DATASET}/ecg_windowed.norm.csv"
DATA_EDA = f"{DATASET}/eda_windowed.norm.csv"
DATA_FS = 500 # Hz
DATA_WINDOW_DURATION = 60 # seconds
TARGET_FS = 51.2
RANDOM_STATE = 21

# Training parameters
HOLDOUT_ITERATIONS = 10
HOLDOUT_TEST_PROPORTION = 0.2
KF_N_FOLDS = 5

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": False,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": False,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}


@dataclass
class Dataset:
    X: list[pd.Series]
    y: pd.Series
    groups: np.ndarray[int]

CWT: TypeAlias = Tuple[np.ndarray[tuple[int], np.dtype], np.ndarray]


### Creating test and train images

In [ ]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
labels: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        labels[key] = labels_df[conf["col_name"]]

display(labels['b'])

In [ ]:
# Pairing labels and samples for each class type

raw_num_samples = DATA_FS * DATA_WINDOW_DURATION
raw_df = pd.read_csv(DATA_EDA)

datasets: dict[str, Dataset] = {}
for class_type, labels_set in labels.items():
    subject_to_group: dict[str, int] = {}
    group_counter = 0
    samples: list[pd.Series] = []
    groups_list: list[int] = []
    dataset_labels = pd.Series([], dtype=np.int64, name=labels_set.name)
    for col_name, label in labels_set.items():
        subject_id = col_name.split("_")[0]
        if col_name in raw_df.columns: # Only add labels with corresponding data, EDA dataset lacks one entry which ECG has
            samples.append(raw_df[col_name])
            dataset_labels[col_name] = label
            if subject_id not in subject_to_group:
                subject_to_group[subject_id] = group_counter
                group_counter += 1
            groups_list.append(subject_to_group[subject_id])
    groups = np.array(groups_list)
    datasets[class_type] = Dataset(samples, dataset_labels, groups)
    print(f"Class type: {class_type}. Saving {len(datasets[class_type].X)} samples with {len(datasets[class_type].y)} labels")
    print(f"{len(subject_to_group)} groups found. Indexed {len(groups)}.")

In [ ]:
#display(datasets['b'].groups)
#display(datasets["b"].X)
#display(datasets['b'].y)

### Preprocessing functions

For window length: 10s
The samples per window if 64Hz would be 640 (64 x 10)
Consider 1 second stride as paper: stride samples would be 64

Maintaining the same window length of 10 seconds
Training dataset @500Hz: window(5000 samples), stride(500 samples)
Experiment unseen data @51.2Hz: window(512 samples), stride(51 samples)

To maintain the same image resolution
* a) Data could be downsampled 500Hz->100Hz or upsampled 51.2Hz->100Hz to achieve consistent 640 samples per window
* b) CWT generated images could be resized to a common resolution

In [ ]:
def segment_signal(
    signal: np.ndarray, fs: float, window_size_sec: float = 10, stride_sec: float = 1, target_fs: float | None = None
) -> np.ndarray:
    num_samples = len(signal)
    if target_fs and target_fs != fs:
        target_num_samples = int(num_samples * target_fs / fs)
        signal = resample(signal, target_num_samples)
        fs = target_fs
        num_samples = len(signal)

    window_size = int(window_size_sec * fs)
    stride_size = int(stride_sec * fs)
    segments = []
    end_idx = num_samples - window_size
    end_idx = 1 if end_idx < 1 else end_idx # avoid empty result when sample size is the same as window size
    for start in range(0, end_idx, stride_size):
        segment = signal[start : start + window_size]
        segments.append(segment)
    return np.array(segments)


def cwt_transform(
    signal: np.ndarray, wavelet="cmor1.5-1.0", fs: float = TARGET_FS, freq_band: Tuple[float, float] = (0.01, 5.0), n_bins: int=100
) -> CWT:
    dt = 1.0 / fs
    fmin, fmax = freq_band
    freqs = np.linspace(fmin, fmax, num=n_bins)
    scales = pywt.central_frequency(wavelet) / (freqs * dt)
    coefficients, freqs = pywt.cwt(signal, scales, wavelet, sampling_period=dt)
    return np.abs(coefficients) ** 2, freqs


def create_cwt_images(
    segments: list[np.ndarray], img_size: tuple[int, int] = (128, 128), fs: float = TARGET_FS, freq_band: Tuple[float, float] = (0.01, 5.0)
) -> list[CWT]:
    images: list[CWT] = []
    for seg in segments:
        cwt_img, freqs = cwt_transform(seg, fs=TARGET_FS, freq_band=freq_band)
        cwt_img_resized = resize(cwt_img, img_size, anti_aliasing=True)
        images.append((cwt_img_resized, freqs))
    return images


datagen = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=10, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True
)

### CNN Model

In [ ]:
def build_model(input_shape):
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation="relu", input_shape=input_shape),
            MaxPooling2D((2, 2)),
            Conv2D(64, (3, 3), activation="relu"),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation="relu"),
            # Dense(2, activation="softmax"),
            Dense(1, activation="sigmoid"),  # single probability output
        ]
    )
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    return model

# Suppose you have: X_train (N, H, W), y_train
# Expand dims for channels
# X_train = np.expand_dims(X_train, -1)

# model = build_model(input_shape=(H, W, 1))
# history = model.fit(datagen.flow(X_train, y_train, batch_size=32), epochs=5)

def build_onehead_cnn(eda_shape):
    """
    One-head CNN for stress classification using only EDA CWT images.

    Parameters
    ----------
    eda_shape : tuple
        Shape of EDA spectrogram input (time, freq, 1).
    """

    # Input: CWT spectrogram (image)
    eda_input = Input(shape=eda_shape, name="eda_input")

    # Convolutional feature extractor
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(eda_input)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.GlobalAveragePooling2D()(x)  # summarize spatial features

    # Fully connected classifier
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    # output = layers.Dense(2, activation="softmax")(x)
    output = layers.Dense(1, activation="sigmoid")(x)  # single probability output

    # Model
    model = models.Model(inputs=eda_input, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    return model


### Training and Validation (GroupKFold)

In [ ]:
def show_confusion_matrix(
    cm: np.ndarray, labels: list[int], save=False
):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap=plt.cm.Blues, colorbar=False)
    #if save:
    #    print(f"Saving confusion matrix to {cm_file}")
    #    plt.savefig(cm_file, dpi=300, format="png")
    #plt.show()

In [ ]:
# Create CWT images
CWT_SHOW = True
RESAMPLING_SHOW = True
SHOW_N_SAMPLES = 5
DO_BREAK = True  # Whether to stop or not after SHOW_N_SAMPLES
SHOW_NROWS = (SHOW_N_SAMPLES * 1 if CWT_SHOW else 0) + (SHOW_N_SAMPLES * 2 if RESAMPLING_SHOW else 0)
SHOWS_NCOLS = 1
FIGSIZE = (6, 4 * SHOW_NROWS)
if RESAMPLING_SHOW or CWT_SHOW:
    fig, axes = plt.subplots(SHOW_NROWS, SHOWS_NCOLS, figsize=FIGSIZE)

dt = 1.0 / TARGET_FS
fmin, fmax = (0.01, 2.0)
freqs = np.linspace(fmin, fmax, num=100)
central_freq = pywt.central_frequency("cmor1.5-1.0")
scales = central_freq / (freqs * dt)

X = {}
ax_counter = 0
for class_type, dataset in datasets.items():
    print(f'CWTs for "{class_type}" classes')
    cwts_list: list[np.ndarray[tuple[int], np.dtype]] = []
    i = 0
    for sample in dataset.X:
        sample_name = dataset.y.index[i]
        segments = segment_signal(
            sample.to_numpy(),
            fs=DATA_FS,
            window_size_sec=DATA_WINDOW_DURATION,
            stride_sec=DATA_WINDOW_DURATION,
            target_fs=TARGET_FS,
        )
        smoothed = medfilt(segments[0], kernel_size=11)
        if RESAMPLING_SHOW:
            axes[ax_counter].set_title(f"Original {sample_name}")
            axes[ax_counter].plot(sample, label=f"Original ({DATA_FS}Hz)")
            #axes[ax_counter].set_ylabel("Sample")
            #axes[ax_counter].set_xlabel("Amplitude")
            ax_counter += 1
            axes[ax_counter].set_title("Processed signal")
            axes[ax_counter].plot(smoothed, label=f"Resampled ({TARGET_FS}Hz)")
            #axes[ax_counter].set_ylabel("Sample")
            #axes[ax_counter].set_xlabel("Amplitude")
            ax_counter += 1

        cwts = create_cwt_images([smoothed,], fs=TARGET_FS, freq_band=(fmin, fmax))
        cwt, cwt_freqs = cwts[0]
        cwt_db = 10 * np.log10(cwt + 1e-12) # Log power
        cwts_list.append(cwt_db)

        if CWT_SHOW:
            norm_cwt = cwt_db - np.percentile(cwt_db, 5)
            vmax = np.percentile(norm_cwt, 99)
            vmin = np.percentile(norm_cwt, 1)
            im = axes[ax_counter].imshow(
                norm_cwt,
                extent=[freqs.min(), freqs.max(), cwt_freqs.min(), cwt_freqs.max()],
                cmap="jet",
                aspect="auto",
                origin="lower",
                vmax=vmax,
                vmin=vmin,
            )
            fig.colorbar(im, label="Power (dB)", ax=axes[ax_counter])
            axes[ax_counter].set_ylabel("Frequency (Hz)")
            axes[ax_counter].set_xlabel("Time (s)")
            ax_counter += 1

        i += 1
        if (RESAMPLING_SHOW or CWT_SHOW) and (i+1) > SHOW_N_SAMPLES:
            RESAMPLING_SHOW = CWT_SHOW = False
            if DO_BREAK:
                break

    print(f"Using {len(cwts_list)} samples")
    # Dataset z-score normalization
    power_arr = np.array(cwts_list)
    mean = np.mean(power_arr)
    std = np.std(power_arr)
    power_norm = (power_arr - mean) / std
    X[class_type] = power_norm[..., np.newaxis]  # extra dimension for CNN input


In [ ]:
import os
import shutil
if os.path.exists(f"{LOGS_DIR}/train"):
    shutil.rmtree(f"{LOGS_DIR}/train")
if os.path.exists(f"{LOGS_DIR}/validation"):
    shutil.rmtree(f"{LOGS_DIR}/validation")

In [ ]:
# Group K-fold training and validation
# from livelossplot.tf_keras import PlotLossesCallback
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=LOGS_DIR)
EPOCHS = 30

for class_type, dataset in datasets.items():
    X_cwts = X[class_type]
    print(f"Training for \"{class_type}\" classes")

    stratify = LABELS_CONF[class_type]["stratification"]
    df_res = pd.DataFrame({"n": [], "f1-score": [], "accuracy": [], "classifier": [], "time": []})
    classes_names = LABELS_CONF[class_type]["classes"]

    results = []
    cms = []
    if stratify:
        gkf = StratifiedKFold(n_splits=KF_N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    else:
        gkf = GroupKFold(n_splits=KF_N_FOLDS)
    for i, (train_idxs, test_idxs) in enumerate(gkf.split(X_cwts, dataset.y, dataset.groups)):
        print(f"Group fold {i+1:2d}/{KF_N_FOLDS:2d}")
        X_train = X_cwts[train_idxs]
        y_train = dataset.y[train_idxs]
        X_test = X_cwts[test_idxs]
        y_test = dataset.y[test_idxs]

        model = build_model(input_shape=X_train.shape[1:])
        # model = build_onehead_cnn(X_train.shape[1:])
        if i == 0:
            display(model.summary())

        train_generator = datagen.flow(X_train, y_train, batch_size=32)
        steps = train_generator.n // train_generator.batch_size
        history = model.fit(
            train_generator,
            steps_per_epoch=steps,
            epochs=EPOCHS,
            validation_data=(X_test, y_test),
            #callbacks=[PlotLossesCallback()],
            callbacks=[tensorboard_cb]
        )
        acc = model.evaluate(X_test, y_test, verbose=0)[1]
        results.append(acc)

        # Get a confusion matrix for the last fold
        if (i+1) == KF_N_FOLDS:
            probs = model.predict(X_test)
            y_pred = [1 if y > 0.5 else 0 for y in probs]
            cm = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, target_names=classes_names, output_dict=True)

    # for n_iter in range(HOLDOUT_ITERATIONS):
    #    int_state = random.randint(0, 1000000)
    #    x_train, x_test, y_train, y_test = train_test_split(
    #        X_cwts, dataset.y, test_size=HOLDOUT_TEST_PROPORTION, random_state=int_state, stratify=stratify
    #    )
    #    print("Random Split {0:2d}/{1:2d}".format(n_iter + 1, HOLDOUT_ITERATIONS))
    show_confusion_matrix(cm=cm, labels=classes_names)
    print(report)
    print(f"GroupKFold accuracies per subject: {results}")
    print(f"Mean acc: {np.mean(results):.3f} ± {np.std(results):.3f}")